# Design a Metrics Collection and Analysis System

**Company:** MongoDB (GothamLoop question bank) · **Category:** System Design · **Tags:** Onsite Loop, Caching, Concurrency, Data Engineering, Databases, Distributed Systems · **Difficulty/Frequency:** Uncommon (3/10)

> **Related:** [`2. Distributed_Task_Scheduler`](../2.%20Distributed_Task_Scheduler/2.%20Distributed_Task_Scheduler.ipynb) makes the same partition-drop argument from the scheduler's side.

## Concepts

**What this question is really testing:** whether you can *size* a system before designing it. Every structural decision here — tiering, rollups, partition drops — is forced by one number, and the number comes first.

**First-principles primer:**

- **A time series** is `(what, when, value)` repeated forever. The "forever" is the whole problem: the data arrives at a constant rate and is never updated, so the table only grows, and every query is a *range scan* over time.
- **Pull vs. push.** *Push* means the client sends to you: simple for the client (no inbound firewall rules), but a misbehaving client can flood you and a lost batch is gone. *Pull* means you fetch from the client: you control the rate, you can retry, and a dead client just stops answering — which is itself a useful signal. Pull's weakness is reachability (NAT, short-lived instances), so the production answer is **pull with a push gateway as an escape hatch**.
- **A rollup** is a pre-computed summary of a time bucket. Instead of scanning 240 raw samples to draw one point on an hourly chart, you read one pre-aggregated row. You are trading storage and freshness for query speed.
- **Partition drop vs. `DELETE`.** `DELETE FROM metrics WHERE ts < ...` on billions of rows rewrites the table, bloats the index, and takes hours. `DROP TABLE metrics_recent_2025_01_15` unlinks a file. Same effect, **O(1) instead of O(rows)** — which is why the table is partitioned by day in the first place.

**Simple worked example.** One instance, one metric, scraped every 15 seconds:

```
raw:      09:00:00  42      Keep for 7 days.
          09:00:15  47      240 rows per hour.
          09:00:30  41
          ...                        ↓  rollup job, hourly
1h rollup: bucket=09:00  count=240  min=38  max=61  avg=44.2  p95=57
                                    ↑ one row replaces 240

1d rollup: bucket=2025-01-15  count=5760  min=31  max=88  avg=45.1  p95=59
                                    ↑ one row replaces 5,760
```

A dashboard showing "last 30 days" reads **30 rows** from the daily rollup, not 172,800 raw samples. That factor *is* the design.

**The one asymmetry to remember:** metrics are **lossy by design**. If the write path saturates, you drop samples — you never apply backpressure to the client database. Its availability matters more than your telemetry's completeness. Almost nothing else in systems design is allowed to say that, and saying it out loud is what marks this as a metrics problem rather than a generic ingest problem.

## Requirements & Scale

| Functional | Non-functional |
|---|---|
| Periodically collect operational + performance metrics | Scale to thousands of client instances |
| Store time-series data **and** metadata | Graceful degradation, not perfect delivery |
| Query: aggregations, dashboards, alerting | Per-client, plan-tier retention |
| Support push **and** pull collection | Retention enforcement must be cheap |

**Stated scale:** 5,000 client DB instances × 200 metrics, scraped every 15 seconds.

In [ ]:
import os, sys
_root = os.getcwd()
for _ in range(5):
    if os.path.exists(os.path.join(_root, "capacity.py")):
        break
    _root = os.path.dirname(_root)
if _root not in sys.path:
    sys.path.insert(0, _root)

from capacity import (SECOND, MINUTE, HOUR, DAY, YEAR, KB, MB, GB, TB,
                      human_bytes, human_count, human_rate, table,
                      assumption_table, sensitivity)

INSTANCES = 5_000
METRICS_PER_INSTANCE = 200
SCRAPE_INTERVAL_SEC = 15

BYTES_VALUE, BYTES_TAGS, BYTES_TS = 8, 50, 12
BYTES_PER_SAMPLE = BYTES_VALUE + BYTES_TAGS + BYTES_TS      # 70

ROLLUP_ROW_BYTES = 100
RAW_RETENTION_DAYS = 7
ROLLUP_1H_RETENTION_DAYS = 90
ROLLUP_1D_RETENTION_DAYS = 395          # "13 months"

DASHBOARD_USERS = 100
DASHBOARD_REFRESH_SEC = 5
QUERIES_PER_REFRESH = 10

assumption_table({
    "Client DB instances":   human_count(INSTANCES),
    "Metrics per instance":  METRICS_PER_INSTANCE,
    "Scrape interval":       f"{SCRAPE_INTERVAL_SEC} s",
    "Bytes per sample":      f"{BYTES_PER_SAMPLE} B  ({BYTES_VALUE} value + {BYTES_TAGS} tags + {BYTES_TS} ts)",
    "Rollup row size":       f"{ROLLUP_ROW_BYTES} B",
    "Raw retention":         f"{RAW_RETENTION_DAYS} days",
    "1h rollup retention":   f"{ROLLUP_1H_RETENTION_DAYS} days",
    "1d rollup retention":   f"{ROLLUP_1D_RETENTION_DAYS} days (13 months)",
})

## Ingest — the number the whole design hangs on

In [ ]:
def bps(n):                       # a BYTE rate; human_rate() is for counts
    return human_bytes(n) + "/s"

samples_per_sec = INSTANCES * METRICS_PER_INSTANCE / SCRAPE_INTERVAL_SEC
bytes_per_sec = samples_per_sec * BYTES_PER_SAMPLE
raw_per_day = bytes_per_sec * DAY

table([
    ("Ingest rate",              human_rate(samples_per_sec, " samples/s")),
    ("Ingest bandwidth",         bps(bytes_per_sec)),
    ("Raw storage / day",        human_bytes(raw_per_day)),
    (f"Raw hot set ({RAW_RETENTION_DAYS}d)", human_bytes(raw_per_day * RAW_RETENTION_DAYS)),
    ("", ""),
    ("Rows / day",               human_count(samples_per_sec * DAY)),
    ("Rows / 30 days",           human_count(samples_per_sec * DAY * 30)),
], title="INGEST")

assert 66_000 < samples_per_sec < 67_000, "the answer's stated ~66,700 samples/s"
assert abs(raw_per_day - 403 * GB) / GB < 5, "the answer's stated ~403 GB/day"
assert abs(raw_per_day * RAW_RETENTION_DAYS - 2.8 * TB) / TB < 0.05, "stated ~2.8 TB"
assert abs(samples_per_sec * DAY * 30 - 172e9) / 1e9 < 2, "stated ~172 billion rows/month"

### ⚠️ Correction 1 — the bandwidth figure is 100× too large

> *"With tags and overhead, ~100 KB/s per instance, or ~500 MB/s total."*

Both numbers are wrong, and the answer **contradicts itself two bullets later**: 403 GB/day works out to 4.67 MB/s, not 500 MB/s. Had ingest really been 500 MB/s, raw storage would be 43 TB/day.

In [ ]:
per_instance = METRICS_PER_INSTANCE / SCRAPE_INTERVAL_SEC * BYTES_PER_SAMPLE
implied_by_storage = raw_per_day / DAY               # derived from the 403 GB/day figure
CLAIMED_PER_INSTANCE, CLAIMED_TOTAL = 100 * KB, 500 * MB

table([
    ("Per instance, stated",  bps(CLAIMED_PER_INSTANCE)),
    ("Per instance, actual",  bps(per_instance)),
    ("  overstated by",       f"{CLAIMED_PER_INSTANCE / per_instance:.0f}x"),
    ("", ""),
    ("Fleet total, stated",   bps(CLAIMED_TOTAL)),
    ("Fleet total, actual",   bps(bytes_per_sec)),
    ("  overstated by",       f"{CLAIMED_TOTAL / bytes_per_sec:.0f}x"),
], title="CORRECTION 1: BANDWIDTH")

assert abs(per_instance - 933) < 2, "200 metrics / 15s x 70 B = 933 B/s, not 100 KB/s"
assert 105 < CLAIMED_PER_INSTANCE / per_instance < 115
assert 105 < CLAIMED_TOTAL / bytes_per_sec < 115

# The internal contradiction: the two stated figures cannot both be true.
assert abs(implied_by_storage - bytes_per_sec) < 1, "403 GB/day IS 4.67 MB/s"
table([
    ("403 GB/day implies",          bps(implied_by_storage)),
    ("500 MB/s would imply",        human_bytes(CLAIMED_TOTAL * DAY) + "/day"),
    ("...vs the stated",            human_bytes(raw_per_day) + "/day"),
], title="THE TWO STATED FIGURES CONTRADICT EACH OTHER")
assert CLAIMED_TOTAL * DAY > 40 * TB, "500 MB/s = 43 TB/day, 107x the stated storage"

print("\n  => Why it matters: at 4.67 MB/s a single 1 Gb NIC is 96% idle and bandwidth")
print("     never enters the design. At 500 MB/s you would need a dedicated ingest")
print("     fleet and wire compression. The wrong number sends the whole interview")
print("     down a road the system does not need.")
NIC_1GBPS = 125 * MB
print(f"\n     1 Gb/s NIC utilisation at the real rate: {bytes_per_sec / NIC_1GBPS:.1%}")

## Rollups — why the cold tier exists

In [ ]:
samples_per_hour = HOUR / SCRAPE_INTERVAL_SEC
samples_per_day = DAY / SCRAPE_INTERVAL_SEC
series = INSTANCES * METRICS_PER_INSTANCE          # distinct (instance, metric) pairs

rollup_1h_per_day = series * 24 * ROLLUP_ROW_BYTES
rollup_1d_per_day = series * 1 * ROLLUP_ROW_BYTES

table([
    ("Distinct series",            human_count(series)),
    ("", ""),
    ("1h rollup / day",            human_bytes(rollup_1h_per_day)),
    (f"  over {ROLLUP_1H_RETENTION_DAYS} days",
     human_bytes(rollup_1h_per_day * ROLLUP_1H_RETENTION_DAYS)),
    ("1d rollup / day",            human_bytes(rollup_1d_per_day)),
    (f"  over {ROLLUP_1D_RETENTION_DAYS} days",
     human_bytes(rollup_1d_per_day * ROLLUP_1D_RETENTION_DAYS)),
], title="ROLLUP STORAGE")

assert abs(rollup_1h_per_day - 2.4 * GB) / GB < 0.01, "stated 2.4 GB/day"
assert abs(rollup_1d_per_day - 100 * MB) / MB < 1, "stated 100 MB/day"
assert abs(rollup_1h_per_day * 90 - 216 * GB) / GB < 1, "stated ~216 GB"
assert abs(rollup_1d_per_day * 395 - 39.5 * GB) / GB < 1, "stated ~39 GB"

# The whole cold tier costs less than a day of raw data.
cold_total = rollup_1h_per_day * ROLLUP_1H_RETENTION_DAYS + rollup_1d_per_day * ROLLUP_1D_RETENTION_DAYS
table([
    ("13 months of rollups",  human_bytes(cold_total)),
    ("ONE day of raw",        human_bytes(raw_per_day)),
    ("Ratio",                 f"{cold_total / raw_per_day:.2f} : 1"),
], title="THE BARGAIN")
print("\n  => Over a year of queryable history costs less than 3 days of raw samples.")

### ⚠️ Correction 2 — the reduction factors are computed at the wrong interval

> *"Rollups cut the query surface by 360× for hourly data and 8,640× for daily data."*

360 and 8,640 are the factors for a **10-second** interval. At the 15-second interval the answer actually specifies, they are **240×** and **5,760×**.

The conclusion survives — 240× still justifies the rollup tier — but the reduction factor *is* the justification, so it has to follow from the interval you named. Derive it; don't quote it.

In [ ]:
STATED_1H, STATED_1D = 360, 8_640

table([
    ("1h reduction, actual",   f"{samples_per_hour:,.0f}x"),
    ("1h reduction, stated",   f"{STATED_1H:,}x"),
    ("1d reduction, actual",   f"{samples_per_day:,.0f}x"),
    ("1d reduction, stated",   f"{STATED_1D:,}x"),
], title="CORRECTION 2: ROLLUP REDUCTION")

assert samples_per_hour == 240 and samples_per_day == 5_760
assert HOUR / STATED_1H == 10 and DAY / STATED_1D == 10, \
    "the stated factors are exactly right for a 10-second interval"
print(f"\n  => The stated factors imply a {HOUR / STATED_1H:.0f}-second scrape interval,")
print(f"     not the {SCRAPE_INTERVAL_SEC}-second one the answer specifies.")

# Derive it instead of quoting it, and the number tracks the assumption.
print()
for interval in (1, 5, 10, 15, 30, 60):
    print(f"    {interval:>3}s interval  ->  1h rollup collapses {HOUR // interval:>5,} samples,"
          f"   1d collapses {DAY // interval:>6,}")

# What the reduction actually buys: a "last 30 days" dashboard panel.
raw_rows = 30 * samples_per_day
table([
    ("Reading raw",     f"{raw_rows:,.0f} rows"),
    ("Reading 1h",      f"{30 * 24:,} rows"),
    ("Reading 1d",      f"{30:,} rows"),
], title='A "LAST 30 DAYS" PANEL, ONE SERIES')
assert raw_rows / 30 == samples_per_day
print(f"\n  => {raw_rows / 30:,.0f}x fewer rows. And 30 rows is all a 30-point chart can draw.")

## Retention: why partition drops, not `DELETE`

`DELETE FROM metrics_recent WHERE ts < now() - interval '7 days'` looks like the obvious way to enforce retention. On this data volume it is a production incident:

- it rewrites every row it touches and leaves the space needing a vacuum,
- it bloats the index rather than shrinking it,
- it takes a long transaction, on the table taking 66,700 inserts a second.

`DROP TABLE metrics_recent_2025_01_08` unlinks the files. Same effect, constant time, no vacuum, no index churn — and it is the reason the table is `PARTITION BY RANGE (ts)` at all. **Partitioning here is not a performance tweak; it is the delete strategy.**

In [ ]:
rows_per_partition = samples_per_sec * DAY

table([
    ("Rows in one daily partition", human_count(rows_per_partition)),
    ("Bytes in one partition",      human_bytes(raw_per_day)),
    ("", ""),
    ("DELETE cost",                 f"O(n) = {rows_per_partition:,.0f} row rewrites + vacuum"),
    ("DROP PARTITION cost",         "O(1) = one unlink"),
], title="RETENTION ENFORCEMENT")

# Rollups DO need DELETE (they are not partitioned by day here) - but they are 168x smaller.
rollup_rows_per_day = series * 24
print(f"\n  Rollups still need batched DELETEs, but at {rollup_rows_per_day:,.0f} rows/day")
print(f"  instead of {rows_per_partition:,.0f} - {rows_per_partition / rollup_rows_per_day:,.0f}x smaller,")
print("  which is what makes a DELETE tolerable there and intolerable here.")
assert rows_per_partition / rollup_rows_per_day > 100

## Query tier and collector fleet

In [ ]:
# --- Query load ---
dashboard_qps = DASHBOARD_USERS * QUERIES_PER_REFRESH / DASHBOARD_REFRESH_SEC
assert dashboard_qps == 200, "the answer's stated 200 QPS"

# --- Collector fleet ---
TARGETS_PER_COLLECTOR = 10_000        # the answer's figure, at a 30s interval
collectors_needed = INSTANCES / TARGETS_PER_COLLECTOR
scrapes_per_sec = INSTANCES / SCRAPE_INTERVAL_SEC

table([
    ("Dashboard query load",        human_rate(dashboard_qps, " QPS")),
    ("", ""),
    ("Scrapes / s (fleet)",         f"{scrapes_per_sec:,.0f}"),
    ("Collectors for capacity",     f"{collectors_needed:.1f}"),
    ("Collectors for AVAILABILITY", "3  (N+1, sharded by client_id hash)"),
], title="QUERY + COLLECTION")

print("\n  => One collector covers the whole fleet twice over. You run three anyway,")
print("     because a stateless sharded fleet turns a node failure into a re-shard.")
print("     Capacity is not the reason for the second and third node.")
assert collectors_needed < 1, "capacity is not what sizes the collector fleet here"

## Sensitivity — which assumption is actually load-bearing

A capacity estimate is worth having only if you know which input it hangs on. Here the answer to "what would blow this up?" is not obvious from the architecture diagram.

In [ ]:
def raw_storage(instances=INSTANCES, metrics=METRICS_PER_INSTANCE,
                interval=SCRAPE_INTERVAL_SEC, sample_bytes=BYTES_PER_SAMPLE):
    return instances * metrics / interval * sample_bytes * DAY

base = raw_storage()
sensitivity(lambda m: raw_storage(instances=INSTANCES * m), 1.0,
            "any single input", fmt=human_bytes)

# Storage is a product of four linear terms, so scaling ANY of them by m scales the
# result by m. Assert that rather than printing four identical tables.
for scaled in (lambda m: raw_storage(instances=INSTANCES * m),
               lambda m: raw_storage(metrics=METRICS_PER_INSTANCE * m),
               lambda m: raw_storage(interval=SCRAPE_INTERVAL_SEC / m),
               lambda m: raw_storage(sample_bytes=BYTES_PER_SAMPLE * m)):
    for m in (0.5, 2, 5, 10):
        assert abs(scaled(m) - base * m) < 1, "every input is exactly linear"

print("\n  => Instances, metrics/instance, scrape FREQUENCY and bytes/sample all")
print("     produce that identical curve - storage is a product of four linear")
print("     terms, so no single input dominates and none is safe to hand-wave.")
print("     But note WHO controls them: three of the four are set by the CLIENT.")
print("     That is the cardinality-explosion follow-up - a customer adding a")
print("     high-cardinality label multiplies 'metrics per instance' without asking.")

# Per-client budgets are the defence. What does one bad tenant cost?
one_client_metrics = METRICS_PER_INSTANCE
exploded = one_client_metrics * 100        # a label with 100 distinct values
table([
    ("Normal client, series",        human_count(one_client_metrics)),
    ("After one 100-value label",    human_count(exploded)),
    ("Extra raw storage / day",      human_bytes(
        (exploded - one_client_metrics) / SCRAPE_INTERVAL_SEC * BYTES_PER_SAMPLE * DAY)),
    ("As % of the WHOLE fleet",      f"{(exploded - one_client_metrics) / (INSTANCES * METRICS_PER_INSTANCE):.1%}"),
], title="CARDINALITY EXPLOSION: ONE CLIENT, ONE BAD LABEL")
print("\n  => A single tenant adding one 100-value label costs ~2% of total fleet")
print("     storage. Ten of them cost 20%. Hence per-client metric budgets and")
print("     label-cardinality limits enforced at registration, not at query time.")

## Discussion — the follow-ups

- **Cardinality explosion.** The cell above prices it: one tenant, one 100-value label, ~2% of fleet storage. Defences in order of preference: a **cardinality limit at registration** (reject the metric definition, so the client learns immediately), a **per-client series budget** enforced at ingest (drop new series past the cap, keep existing ones), and sampling as a last resort. Rejecting at write time is far better than discovering it in a query timeout.
- **Alerting on gaps.** A threshold rule can only fire on data that arrived. If collection fails, `cpu > 90%` is silently *true of nothing* — the most dangerous failure mode a monitoring system has. You need **staleness detection**: alert on `absent(metric) for 5m` as a first-class rule type, and make it the default for every registered instance rather than something each user remembers to add.
- **Multi-tenancy with strict isolation.** Shard storage by `client_id` so a tenant's data is physically separable (which is also what makes deletion-on-request tractable), scope every query at the API layer rather than trusting the query string, and consider per-tenant encryption keys if the requirement is contractual. The trap is filtering by tenant *after* the scan — that is a performance bug and a security bug at once.
- **User-defined metrics.** A registration API that validates name, type, unit, and — critically — declares which labels are allowed and how many values each may take. Dynamic metrics without a schema is how cardinality explosions happen.
- **Backfilling a 6-hour collector outage.** Pull-based collection means the samples were never sent, so there is nothing to replay unless the *client agent* buffered them — which is the argument for a small local ring buffer plus a `?since=` parameter on the scrape endpoint. Failing that, the hour is permanently at reduced fidelity, and the honest answer is to mark the gap rather than interpolate over it. Interpolated data that looks real is worse than a visible hole.

## Patterns learned

- **Size it before you design it.** 66,700 samples/s and 403 GB/day are what force tiering, rollups, and partition drops. Present the arithmetic first and every structural decision defends itself.
- **Derive numbers, don't quote them.** Both errors here came from a figure that stopped tracking its assumption — a bandwidth number 100× off and contradicting the storage number beside it, and reduction factors still computed at an interval the design no longer used.
- **When two of your own numbers disagree, one of them is a bug.** 500 MB/s and 403 GB/day cannot both be true. Cross-checking derived figures against each other catches errors nothing else will.
- **Separate the write path from the read path.** Append-only raw ingest, pre-aggregated rollups for queries. They have nothing in common but the data.
- **Make retention a schema decision.** The table is partitioned by day *so that* deletion is an unlink. Partitioning here isn't tuning, it's the delete strategy.
- **Keep slow-changing relational data out of the TSDB.** Metric definitions and client info need transactions and are tiny; samples need throughput and are enormous. Mixing them makes each store do the thing it is worst at.
- **Metrics are lossy by design.** Drop samples under overload; never backpressure the system you are observing. The observed system's availability outranks your telemetry.
- **Alert on absence, not just on thresholds.** A rule that only fires on data cannot fire when collection dies — which is exactly when you need it.
- **Capacity is not always what sizes a fleet.** One collector covers 5,000 targets twice over; you run three for availability. Say which constraint you are sizing against.